### Setting up GLINT MEMs

Computer: scexao5

Serial: 32AW038#027



#### SCExAO Method
1. Attach to bmc tmux: 

        `tmux a -t bmc`

2. Run special Vince file

        `hwint-bmc111 -g`

3. Outside of the terminal, the following code is run to move each actuator:

In [ ]:
import ImageStreamIOWrap
from pyMilk.interfacing.shm import SHM  
import numpy as np  
import time

class MEMS():
    '''
    Class to control the MEMS DM.
    '''
    def __init__(self) -> None:
        self.dmvolt = SHM('dmvolt')  
        self.dmptt = SHM('dmptt') 
    
    def set_ptt(self, segment:int, piston:float, tip:float, tilt:float) -> None:
        data = self.dmptt.get_data()
        reshaped_data = data.reshape(37, 3)

        reshaped_data[segment, 0] = piston
        reshaped_data[segment, 1] = tip
        reshaped_data[segment, 2] = tilt

        updated_data = reshaped_data.reshape(111)
        self.dmptt.set_data(updated_data)
        
    def set_volt(self, actuator:int, voltage:float) -> None:
        data = self.dmvolt.get_data()
        data[actuator] = voltage
        self.dmvolt.set_data(data)

    def get_volt(self) -> np.ndarray:
        return self.dmvolt.get_data()

    def get_ptt(self) -> np.ndarray:
        return self.dmptt.get_data()

    def flatten_ptt(self) -> None:
        flat = np.zeros(111)
        self.dmptt.set_data(flat)
        
    def flatten_volt(self) -> None:
        
        flat = np.zeros(111)  # Alternatively, can load one of the flat maps e.g. '32AW038_1500nm.txt'
        self.dmvolt.set_data(flat)
    
def pokeall(mirror):
    '''
    Function to poke all actuators and save the pupil and image data.
    '''

    pupil = SHM('glintpg1')
    image = SHM('glintpg1')
    path = '/home/scexao/steph/bmc/mapping_investigation/pokeall_frames/'
        
    num_actuators = len(mirror.get_volt())
    mirror.flatten_volt()
    
    for x in range(num_actuators):
        mirror.set_volt(x, 0.3)
        

        p = pupil.get_data()
        i = image.get_data()
        
        np.savez(path+f'pupil_actuator{x}_volt0.3', p)
        np.savez(path+f'image_actuator{x}_volt0.3', i)

        mirror.flatten_volt()
        time.sleep(0.01)  

if __name__ == "__main__":
    dm = MEMS()
    dm.pokeall(dm)

#### From Scratch

I just followed the DMSDK python documentation found `hardwaresecrets/drivers/bmc111/DMSDK/Documentation/BMC DM-SDK Python Getting Started.pdf`.

1. Copied the contents in the directory `/opt/Boston/lib/Python3/site-packages/bmc` into my own source code directory `/home/scexao/steph/bmc`

2. Also copied LUT file  `/opt/Boston/Calibration/LUT_32AW038#027.mat` to the same source code directory

3. Tested the examples `dm_test.py`, `pokeall.py`, and `tiltsegments.py` provided by Boston in the directory `/opt/Boston/Examples/Python`. All of these examples ran without errors.

4. In each example, I used the serial number to open the DM i.e. `dm.open_dm('32AW038#027')` (previously `dm.open_dm('MultiUSB000')`)

5. In `tiltsegments.py` example, changed the LUT filename from `'Sample_Lookup_Table.mat'` to `'LUT_32AW038#027.mat'`

6. Ran ` visualise actuator mapping (this is just a slightly modified version of the `pokeall.py` example):


In [ ]:
# bmc.py is in parent directory. There's likely a better way to do this.
import sys
sys.path.append('/home/scexao/steph/bmc')

import bmc
import time
import numpy as np
from pyMilk.interfacing.shm import SHM

pupil = SHM('glintpg1')
image = SHM('glintpg1')
path = '/home/scexao/steph/bmc/mapping_investigation/pokeall_frames/'


def pokeall(mirror):
    """
    Poke all actuators using numpy arrays.

    The numpy array must be converted to a list before sending to the DM.
    """
        
    data = np.zeros(dm.num_actuators(), dtype=float)
    mirror.send_data(data)
    
    for x in range(mirror.num_actuators()):

        data[x] = 0.5
        mirror.send_data(data.tolist())

        p = pupil.get_data()
        i = image.get_data()
        np.savez(path+f'pupil_pokeall_actuator{x}_volt0.5', p)
        np.savez(path+f'image_pokeall_actuator{x}_volt0.5', i)

        data[x] = 0.0
        time.sleep(0.01)  
          
    print("Sent ", x+1, " shapes.")
    dm.send_data(data)


dm = bmc.BmcDm()

# SRB: changed serial number from 'MultiUSB000' to match the one on the DM
serial = '32AW038#027'
err_code = dm.open_dm(serial)
if err_code:
    raise Exception(dm.error_string(err_code))

pokeall(dm)

dm.close_dm()